In [9]:
# data

from data.get import get_data
from data.splits import load_splits

In [10]:
defaults = {
    'channels_to_pick': sorted(['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'C4', 'CP5', 'CP1', 'CP2', 'CP6', 'FC3', 'FCz',
                                'FC4', 'C5', 'C1', 'C2', 'C6', 'CP3', 'CPz', 'CP4', 'FFC5h', 'FFC3h', 'FFC4h',
                                'FFC6h', 'FCC5h', 'FCC3h', 'FCC4h', 'FCC6h', 'CCP5h', 'CCP3h', 'CCP4h', 'CCP6h',
                                'CPP5h', 'CPP3h', 'CPP4h', 'CPP6h', 'FFC1h', 'FFC2h', 'FCC1h', 'FCC2h', 'CCP1h',
                                'CCP2h', 'CPP1h', 'CPP2h']),
    'low_cut_hz': 4,
    'hi_cut_hz': 124,
    's_freq': 250,
    'max_abs_val': 800,
    'trial_stop_offset_samples': 0,
    'trial_start_offset_samples': 125,
    'n_jobs': 1,
    'validation_split_point': 0.8,
}

params = {
    'batch_size': 256,
    'shuffle': False,
    'drop_last': False
}

In [11]:
mode='valid'
sub=1
name='Schirrmeister2017'

trainset, testset = load_splits(*get_data(sub, name=name, **defaults), mode=mode, **params)

Creating RawArray with float64 data, n_channels=128, n_times=1225545
    Range : 0 ... 1225544 =      0.000 ...  2451.088 secs
Ready.
Creating RawArray with float64 data, n_channels=128, n_times=616535
    Range : 0 ... 616534 =      0.000 ...  1233.068 secs
Ready.
320 events found
Event IDs: [1 2 3 4]
160 events found
Event IDs: [1 2 3 4]
Used Annotations descriptions: ['feet', 'left_hand', 'rest', 'right_hand']
Adding metadata with 4 columns
Replacing existing metadata with 4 columns
320 matching events found
No baseline correction applied
0 projection items activated
Loading data for 320 events and 875 original time points ...
0 bad epochs dropped
Loading data for 320 events and 875 original time points ...
Used Annotations descriptions: ['feet', 'left_hand', 'rest', 'right_hand']
Adding metadata with 4 columns
Replacing existing metadata with 4 columns
160 matching events found
No baseline correction applied
0 projection items activated
Loading data for 160 events and 875 original 

In [12]:
import torch
import torch.nn as nn

from spdnet.spd import SPDTransform
from eegspdnet.nn import ChSpecConv, SCMPool, AltLogEig, AltReEig
from eegspdnet.optim import AltStiefelMetaOptimizer


In [13]:
n_bands = 4
n_classes = 4
n_ch = len(defaults['channels_to_pick'])

In [18]:
class EEGSPDNet(nn.Module):
    
    def __init__(self, n_ch, n_bands, n_classes):
        super(__class__, self).__init__()
    
        self.conv = ChSpecConv(n_ch=n_ch, n_filters=n_bands, n_time_kernel=25)
        self.scm = SCMPool(n_bands, add_d=False)
        
        bimap_sizes = [(n_bands * n_ch) // 2**i for i in range(4)]
        self.bimap1 = SPDTransform(bimap_sizes[0], bimap_sizes[1])
        self.reeig1 = AltReEig()
        self.bimap2 = SPDTransform(bimap_sizes[1], bimap_sizes[2])
        self.reeig2 = AltReEig()
        self.bimap3 = SPDTransform(bimap_sizes[2], bimap_sizes[3])
        self.reeig3 = AltReEig()
        self.log = AltLogEig(bimap_sizes[-1])
        
        vectorised_size = int(bimap_sizes[-1] * (bimap_sizes[-1]) * 0.5)
        
        self.linear = nn.Linear(vectorised_size, n_classes)
        
    def forward(self, x):
        
        x = self.conv(x)
        x = self.scm(x)
        x = self.bimap1(x)
        x = self.reeig1(x)
        x = self.bimap1(x)
        x = self.reeig1(x)
        x = self.bimap1(x)
        x = self.reeig1(x)
        x = self.log(x)
        x = self.linear(x.flatten(start_dim=1))
        return x
    
model = EEGSPDNet(n_ch=n_ch, n_bands=n_bands, n_classes=n_classes)
_optim = torch.optim.Adam(model.parameters(), lr=1e-3)
optim = AltStiefelMetaOptimizer(_optim)
criterion = nn.CrossEntropyLoss()

In [19]:
n_epochs = 50

# train

for epoch in range(n_epochs):
    
    model.train()
    
    train_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, batch in enumerate(trainset):
        data = batch['data']
        labels = batch['label'].squeeze().long()
        
        optim.zero_grad()
        outputs = model(data)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()
        
        train_loss += loss.data.item()
        _, preds = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += preds.eq(labels.data).cpu().sum().data.item()
        
    final_loss = train_loss / (batch_idx + 1)
    acc = 100 * (correct / total)
    
    print(f'{epoch}: {final_loss:.2f}, {acc:.2f}%')

torch.Size([256, 176, 176])


RuntimeError: Expected batch2_sizes[0] == bs && batch2_sizes[1] == contraction_size to be true, but got false.  (Could this error message be improved?  If so, please report an enhancement request to PyTorch.)